# Part A – Data Loading and Initial Inspection

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("bank-full.csv", sep=";")

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


The Bank Marketing dataset used in this project is the complete dataset stored in the bank-full.csv file. It contains information collected from direct marketing campaigns carried out by a Portuguese banking institution. The data describes bank clients, the way they were contacted, and information from previous marketing campaigns.

In [2]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 45211
Number of columns: 17


The complete dataset contains 45,211 rows and 17 columns. Each row represents a marketing campaign record associated with a bank client. It includes information about the client, the current contact, and previous marketing campaign activity. Sixteen columns are input features that describe the client, the contact process, and previous campaign activity. The remaining column, y, is the prediction target.

In [3]:
print("Column names:")
print(df.columns.tolist())

Column names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


The dataset includes client-related variables, such as age, job, marital status, education, account balance, and loan status. It also contains information about the current contact and previous marketing campaigns. The final column, y, records whether the client subscribed to a term deposit.

In [4]:
target_column = "y"

print("Target column:", target_column)
print("Target values:", df[target_column].unique())

Target column: y
Target values: <StringArray>
['no', 'yes']
Length: 2, dtype: str


The prediction target is the variable `y`. A value of `yes` means that the client subscribed to a term deposit, while a value of `no` means that the client did not subscribe. Therefore, this project is a binary classification problem.

In [5]:
target_column = "y"
input_features = df.drop(columns=[target_column])

numerical_columns = input_features.select_dtypes(include="number").columns.tolist()
categorical_columns = input_features.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numerical input features:")
print(numerical_columns)
print("Number of numerical input features:", len(numerical_columns))

print("\nCategorical input features:")
print(categorical_columns)
print("Number of categorical input features:", len(categorical_columns))

Numerical input features:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Number of numerical input features: 7

Categorical input features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Number of categorical input features: 9


The dataset contains seven numerical input features: age, balance, day, duration, campaign, pdays, and previous. It also contains nine categorical input features: job, marital, education, default, housing, loan, contact, month, and poutcome. The target variable, y, is kept separate from the input features.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


The dataset contains both numerical and categorical variables. Seven columns are stored as integers and ten columns are stored as strings. All 17 columns contain 45,211 non-null values, so no missing values are detected at this stage.

In [7]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


The dataset contains no missing values. All 17 columns have zero null values, so no missing-value treatment is required at this stage.

In [8]:
unknown_counts = (df == "unknown").sum()

print("Unknown values:")
print(unknown_counts[unknown_counts > 0])

Unknown values:
job            288
education     1857
contact      13020
poutcome     36959
dtype: int64


The dataset contains "unknown" values in four categorical columns. The job column contains 288 unknown values, education contains 1,857, contact contains 13,020, and poutcome contains 36,959. These values are not missing values in the technical sense, but they represent unavailable or unspecified information and should be considered during preprocessing.

In [9]:
pdays_minus_one_count = (df["pdays"] == -1).sum()
pdays_minus_one_percentage = pdays_minus_one_count / len(df) * 100

print(f"Rows with pdays = -1: {pdays_minus_one_count:,}")
print(f"Percentage with pdays = -1: {pdays_minus_one_percentage:.2f}%")

Rows with pdays = -1: 36,954
Percentage with pdays = -1: 81.74%


The value -1 appears in the pdays column in 36,954 rows, representing 81.74% of the dataset. According to the dataset description, this value indicates that the client was not previously contacted during an earlier campaign. Therefore, it is a meaningful value rather than a missing value or a data-entry error.

In [10]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


The dataset contains no duplicate rows. Therefore, no duplicate records need to be removed at this stage.

In [11]:
print("Class distribution of y:")
print(df["y"].value_counts())

print("\nClass distribution percentages:")
print(df["y"].value_counts(normalize=True) * 100)

Class distribution of y:
y
no     39922
yes     5289
Name: count, dtype: int64

Class distribution percentages:
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64


The target variable is imbalanced. Approximately 88.30% of the observations belong to the no class, while only 11.70% belong to the yes class. This imbalance should be taken into account during model training and evaluation.

## Part A – Analysis

1. Each row represents a marketing campaign record associated with a bank client. It includes information about the client, the current contact, and previous marketing campaign activity.

2. The prediction target is y, which indicates whether the client subscribed to a term deposit.

3. The target variable is imbalanced. Approximately 88.30% of the observations belong to the no class, while approximately 11.70% belong to the yes class.

4. The dataset contains seven numerical input features: age, balance, day, duration, campaign, pdays, and previous.

5. The dataset contains nine categorical input features: job, marital, education, default, housing, loan, contact, month, and poutcome. The target variable y is kept separate because it is the prediction target rather than an input feature.

6. No conventional missing values or duplicate rows were found. However, the columns job, education, contact, and poutcome contain unknown values. In addition, pdays = -1 appears in 36,954 rows, representing 81.74% of the dataset. According to the dataset description, this value indicates that the client was not previously contacted. Therefore, it is a meaningful value rather than a missing value or a data-entry error.